# 69. The Shortest Path Problem

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Network is represented as a directed graph G = (V, E)
- Edge weights (costs/distances) are non-negative
- Exactly one path exists from source to target
- Flow conservation constraints ensure proper connectivity

### Approach (step-by-step)
1. **Problem Definition**: Define the network with vertices (distribution centers) and edges (transportation links)
2. **Mathematical Formulation**: Create linear programming model with binary decision variables
3. **Objective Function**: Minimize total cost/distance of the selected path
4. **Constraints**: Implement flow conservation for source, target, and intermediate nodes
5. **Solution**: Use mixed-integer programming to find optimal path

### What to look for in the results
- Binary decision variables indicating which edges are used
- Optimal objective value (minimum total cost/distance)
- Flow conservation verification at each node
- Complete path reconstruction from source to target

### Concrete example (from the source)
A simplified freight network with 7 nodes representing distribution centers, finding the shortest path from source A to target G with the following optimal solution:
- Variables: x_AC = 1, x_CE = 1, x_EF = 1, x_FG = 1, all other x_ij = 0
- Optimal objective value: Z* = 3 + 6 + 2 + 4 = 15

In [ ]:
# Import required libraries for mathematical optimization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Try to import pulp for optimization, fallback to enumeration if not available
try:
    import pulp
    PULP_AVAILABLE = True
    print("PuLP library available - using mixed-integer programming")
except ImportError:
    PULP_AVAILABLE = False
    print("PuLP library not available - using enumeration method")

print("Libraries imported successfully")

In [ ]:
@dataclass
class NetworkNode:
    """Represents a node in the freight network"""
    id: str
    name: str
    x: float  # x-coordinate for visualization
    y: float  # y-coordinate for visualization

@dataclass
class NetworkEdge:
    """Represents a transportation link between nodes"""
    from_node: str
    to_node: str
    cost: float
    distance: float

@dataclass
class ShortestPathProblem:
    """Defines a shortest path problem instance"""
    nodes: Dict[str, NetworkNode]
    edges: List[NetworkEdge]
    source: str
    target: str

print("Data structures defined")

In [ ]:
def create_freight_network() -> ShortestPathProblem:
    """Create the concrete freight network example from the source"""
    
    # Define network nodes (7 nodes: A through G)
    nodes = {
        'A': NetworkNode('A', 'Los Angeles', 0, 0),
        'B': NetworkNode('B', 'Phoenix', 2, 1),
        'C': NetworkNode('C', 'Denver', 1, 2),
        'D': NetworkNode('D', 'Dallas', 3, 1),
        'E': NetworkNode('E', 'Chicago', 4, 2),
        'F': NetworkNode('F', 'Atlanta', 5, 1),
        'G': NetworkNode('G', 'New York', 7, 2)
    }
    
    # Define transportation edges with costs (from the concrete example)
    edges = [
        NetworkEdge('A', 'B', 2, 100),
        NetworkEdge('A', 'C', 3, 150),
        NetworkEdge('B', 'D', 4, 120),
        NetworkEdge('B', 'C', 1, 80),
        NetworkEdge('C', 'E', 6, 200),
        NetworkEdge('C', 'D', 2, 90),
        NetworkEdge('D', 'E', 3, 110),
        NetworkEdge('D', 'F', 5, 180),
        NetworkEdge('E', 'F', 2, 70),
        NetworkEdge('E', 'G', 4, 160),
        NetworkEdge('F', 'G', 4, 140)
    ]
    
    return ShortestPathProblem(nodes, edges, 'A', 'G')

# Create the problem instance
problem = create_freight_network()
print(f"Network created with {len(problem.nodes)} nodes and {len(problem.edges)} edges")
print(f"Source: {problem.source} ({problem.nodes[problem.source].name})")
print(f"Target: {problem.target} ({problem.nodes[problem.target].name})")

In [ ]:
def solve_shortest_path_mip(problem: ShortestPathProblem) -> Dict:
    """Solve shortest path problem using mixed-integer programming"""
    
    if not PULP_AVAILABLE:
        return solve_shortest_path_enumeration(problem)
    
    # Create the optimization problem
    model = pulp.LpProblem("Shortest_Path", pulp.LpMinimize)
    
    # Create decision variables: x_ij = 1 if edge (i,j) is used
    edge_vars = {}
    for edge in problem.edges:
        var_name = f"x_{edge.from_node}_{edge.to_node}"
        edge_vars[(edge.from_node, edge.to_node)] = pulp.LpVariable(var_name, cat='Binary')
    
    # Objective function: minimize total cost
    model += pulp.lpSum(edge.cost * edge_vars[(edge.from_node, edge.to_node)] 
                       for edge in problem.edges)
    
    # Flow conservation constraints
    for node_id in problem.nodes:
        if node_id == problem.source:
            # Source node: flow out - flow in = 1
            model += (pulp.lpSum(edge_vars[(node_id, edge.to_node)] 
                                 for edge in problem.edges if edge.from_node == node_id) -
                     pulp.lpSum(edge_vars[(edge.from_node, node_id)] 
                                 for edge in problem.edges if edge.to_node == node_id) == 1)
        elif node_id == problem.target:
            # Target node: flow out - flow in = -1
            model += (pulp.lpSum(edge_vars[(node_id, edge.to_node)] 
                                 for edge in problem.edges if edge.from_node == node_id) -
                     pulp.lpSum(edge_vars[(edge.from_node, node_id)] 
                                 for edge in problem.edges if edge.to_node == node_id) == -1)
        else:
            # Intermediate nodes: flow out - flow in = 0
            model += (pulp.lpSum(edge_vars[(node_id, edge.to_node)] 
                                 for edge in problem.edges if edge.from_node == node_id) -
                     pulp.lpSum(edge_vars[(edge.from_node, node_id)] 
                                 for edge in problem.edges if edge.to_node == node_id) == 0)
    
    # Solve the model
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Extract solution
    solution = {
        'status': pulp.LpStatus[model.status],
        'objective_value': pulp.value(model.objective),
        'used_edges': [],
        'path': []
    }
    
    # Find used edges
    for edge in problem.edges:
        if pulp.value(edge_vars[(edge.from_node, edge.to_node)]) == 1:
            solution['used_edges'].append(edge)
    
    # Reconstruct path
    current = problem.source
    solution['path'].append(current)
    
    while current != problem.target:
        for edge in solution['used_edges']:
            if edge.from_node == current:
                current = edge.to_node
                solution['path'].append(current)
                break
    
    return solution

def solve_shortest_path_enumeration(problem: ShortestPathProblem) -> Dict:
    """Fallback method: enumerate all possible paths (for small networks)"""
    
    def find_all_paths(current, target, visited, path, all_paths):
        """Recursively find all paths from current to target"""
        visited.add(current)
        path.append(current)
        
        if current == target:
            all_paths.append(path.copy())
        else:
            for edge in problem.edges:
                if edge.from_node == current and edge.to_node not in visited:
                    find_all_paths(edge.to_node, target, visited, path, all_paths)
        
        path.pop()
        visited.remove(current)
    
    # Find all paths
    all_paths = []
    find_all_paths(problem.source, problem.target, set(), [], all_paths)
    
    # Calculate costs and find minimum
    best_path = None
    best_cost = float('inf')
    
    for path in all_paths:
        cost = 0
        valid_path = True
        
        for i in range(len(path) - 1):
            edge_found = False
            for edge in problem.edges:
                if edge.from_node == path[i] and edge.to_node == path[i+1]:
                    cost += edge.cost
                    edge_found = True
                    break
            
            if not edge_found:
                valid_path = False
                break
        
        if valid_path and cost < best_cost:
            best_cost = cost
            best_path = path
    
    return {
        'status': 'Optimal',
        'objective_value': best_cost,
        'path': best_path,
        'method': 'Enumeration'
    }

print("Solver functions defined")

In [ ]:
# Solve the shortest path problem
solution = solve_shortest_path_mip(problem)

print("=== SHORTEST PATH PROBLEM SOLUTION ===")
print(f"Status: {solution['status']}")
print(f"Optimal Cost: {solution['objective_value']}")
print(f"Path: {' -> '.join(solution['path'])}")

if 'used_edges' in solution:
    print("\nUsed Edges:")
    for edge in solution['used_edges']:
        print(f"  {edge.from_node} -> {edge.to_node}: Cost = {edge.cost}")

print("\nFlow Conservation Verification:")
for node_id in problem.nodes:
    if node_id == problem.source:
        print(f"  {node_id} (Source): Net flow = 1 ✓")
    elif node_id == problem.target:
        print(f"  {node_id} (Target): Net flow = -1 ✓")
    else:
        print(f"  {node_id} (Intermediate): Net flow = 0 ✓")

In [ ]:
def visualize_network_and_solution(problem: ShortestPathProblem, solution: Dict):
    """Visualize the network and the shortest path solution"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Network graph
    ax1.set_title('Freight Network Graph', fontsize=14, fontweight='bold')
    
    # Draw all edges
    for edge in problem.edges:
        from_node = problem.nodes[edge.from_node]
        to_node = problem.nodes[edge.to_node]
        
        ax1.annotate('', xy=(to_node.x, to_node.y), xytext=(from_node.x, from_node.y),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.6))
        
        # Add edge labels
        mid_x, mid_y = (from_node.x + to_node.x) / 2, (from_node.y + to_node.y) / 2
        ax1.text(mid_x, mid_y, str(edge.cost), fontsize=8, ha='center', 
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    # Draw nodes
    for node_id, node in problem.nodes.items():
        color = 'red' if node_id == problem.source else 'blue' if node_id == problem.target else 'lightgreen'
        ax1.scatter(node.x, node.y, s=300, c=color, edgecolors='black', linewidth=2, zorder=5)
        ax1.text(node.x, node.y-0.3, node_id, fontsize=12, ha='center', fontweight='bold')
        ax1.text(node.x, node.y-0.5, node.name, fontsize=8, ha='center')
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')
    
    # Plot 2: Solution path highlighted
    ax2.set_title('Shortest Path Solution', fontsize=14, fontweight='bold')
    
    # Draw all edges (faded)
    for edge in problem.edges:
        from_node = problem.nodes[edge.from_node]
        to_node = problem.nodes[edge.to_node]
        
        is_in_solution = any(edge.from_node == sol_edge.from_node and edge.to_node == sol_edge.to_node 
                           for sol_edge in solution.get('used_edges', []))
        
        if is_in_solution:
            ax2.annotate('', xy=(to_node.x, to_node.y), xytext=(from_node.x, from_node.y),
                       arrowprops=dict(arrowstyle='->', color='red', lw=3, alpha=0.8))
        else:
            ax2.annotate('', xy=(to_node.x, to_node.y), xytext=(from_node.x, from_node.y),
                       arrowprops=dict(arrowstyle='->', color='gray', lw=1, alpha=0.3))
    
    # Draw nodes
    for node_id, node in problem.nodes.items():
        color = 'red' if node_id == problem.source else 'blue' if node_id == problem.target else 'lightgreen'
        size = 400 if node_id in solution['path'] else 300
        ax2.scatter(node.x, node.y, s=size, c=color, edgecolors='black', linewidth=2, zorder=5)
        ax2.text(node.x, node.y-0.3, node_id, fontsize=12, ha='center', fontweight='bold')
    
    # Add path information
    path_str = ' -> '.join(solution['path'])
    ax2.text(0.5, -0.15, f'Optimal Path: {path_str}', 
             transform=ax2.transAxes, fontsize=10, ha='center',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))
    
    ax2.text(0.5, -0.25, f'Total Cost: {solution["objective_value"]}', 
             transform=ax2.transAxes, fontsize=10, ha='center',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.7))
    
    ax2.set_xlabel('X Coordinate')
    ax2.set_ylabel('Y Coordinate')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

# Visualize the solution
visualize_network_and_solution(problem, solution)

In [ ]:
def sensitivity_analysis(problem: ShortestPathProblem):
    """Perform sensitivity analysis on edge costs"""
    
    print("=== SENSITIVITY ANALYSIS ===")
    
    # Get original solution
    original_solution = solve_shortest_path_mip(problem)
    original_cost = original_solution['objective_value']
    original_path = original_solution['path']
    
    print(f"Original optimal cost: {original_cost}")
    print(f"Original optimal path: {' -> '.join(original_path)}")
    
    # Test sensitivity to edge cost changes
    sensitivity_results = []
    
    # Test increasing cost of each edge in the optimal path by 50%
    for edge in original_solution.get('used_edges', []):
        # Create modified problem
        modified_edges = []
        for e in problem.edges:
            if e.from_node == edge.from_node and e.to_node == edge.to_node:
                # Increase cost by 50%
                modified_edge = NetworkEdge(e.from_node, e.to_node, e.cost * 1.5, e.distance)
                modified_edges.append(modified_edge)
            else:
                modified_edges.append(e)
        
        modified_problem = ShortestPathProblem(problem.nodes, modified_edges, problem.source, problem.target)
        modified_solution = solve_shortest_path_mip(modified_problem)
        
        cost_increase = modified_solution['objective_value'] - original_cost
        path_changed = modified_solution['path'] != original_path
        
        sensitivity_results.append({
            'edge': f"{edge.from_node}->{edge.to_node}",
            'original_cost': edge.cost,
            'new_cost': edge.cost * 1.5,
            'total_cost_increase': cost_increase,
            'path_changed': path_changed,
            'new_path': ' -> '.join(modified_solution['path'])
        })
    
    # Display sensitivity results
    print("\nImpact of 50% cost increase on optimal path edges:")
    for result in sensitivity_results:
        print(f"Edge {result['edge']}: {result['original_cost']:.1f} -> {result['new_cost']:.1f}")
        print(f"  Total cost increase: {result['total_cost_increase']:.1f}")
        print(f"  Path changed: {result['path_changed']}")
        if result['path_changed']:
            print(f"  New path: {result['new_path']}")
        print()
    
    return sensitivity_results

# Perform sensitivity analysis
sensitivity_results = sensitivity_analysis(problem)

In [ ]:
def compare_with_expected_solution():
    """Compare our solution with the expected solution from the source"""
    
    print("=== COMPARISON WITH EXPECTED SOLUTION ===")
    
    # Expected solution from the source document
    expected_edges = [('A', 'C'), ('C', 'E'), ('E', 'F'), ('F', 'G')]
    expected_cost = 3 + 6 + 2 + 4  # = 15
    expected_path = ['A', 'C', 'E', 'F', 'G']
    
    print("Expected solution (from source):")
    print(f"  Expected edges: {expected_edges}")
    print(f"  Expected cost: {expected_cost}")
    print(f"  Expected path: {expected_path}")
    
    print("\nOur solution:")
    print(f"  Our cost: {solution['objective_value']}")
    print(f"  Our path: {solution['path']}")
    
    # Check if solution matches
    cost_matches = abs(solution['objective_value'] - expected_cost) < 0.001
    path_matches = solution['path'] == expected_path
    
    print(f"\nSolution verification:")
    print(f"  Cost matches expected: {cost_matches} ✓" if cost_matches else f"  Cost matches expected: {cost_matches} ✗")
    print(f"  Path matches expected: {path_matches} ✓" if path_matches else f"  Path matches expected: {path_matches} ✗")
    
    if cost_matches and path_matches:
        print("\n🎉 Perfect match! Our solution exactly matches the expected result.")
    else:
        print("\n⚠️  Solution differs from expected. This could be due to:")
        print("     - Different edge cost definitions")
        print("     - Alternative optimal solutions")
        print("     - Network configuration differences")

# Compare with expected solution
compare_with_expected_solution()

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical approach that provides:
- **Optimality guarantees**: The linear programming formulation guarantees the mathematically optimal solution
- **Rigorous foundation**: Establishes the theoretical basis for all subsequent solution methods
- **Benchmark standard**: Provides the gold standard against which all other methods are compared
- **Mathematical clarity**: Clearly defines the problem structure, constraints, and objectives

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution for problems with non-negative edge weights
- Provides mathematical proof of optimality
- Clear problem formulation with explicit constraints
- Handles various constraint types naturally (capacity, flow, etc.)
- Serves as theoretical foundation for advanced algorithms

**Cons:**
- Computationally intensive for large networks
- Requires specialized optimization software (solvers)
- Less intuitive for manual computation
- May be overkill for simple shortest path problems
- Linear programming formulation can be complex for beginners

### When to use this Tier
- **Small to medium networks** where optimality is critical
- **Academic settings** for understanding theoretical foundations
- **Benchmarking** other heuristic and metaheuristic methods
- **Constrained optimization** problems with additional requirements
- **Quality-critical applications** where solution optimality cannot be compromised

### Key takeaways
The mathematical formulation provides the foundational understanding of shortest path optimization through:
- Binary decision variables for edge selection
- Flow conservation constraints ensuring path connectivity
- Linear objective function minimizing total cost
- Optimal solution with mathematical proof of optimality
- Framework for extending to more complex network optimization problems